In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
data = pd.read_csv('../data/processed_shots.csv')
data.head()

In [ ]:
features = ['dist_to_goal', 'angle_to_goal', 'set_shot_flag']
X, y = data[features], data['Score']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Untuned baseline
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Baseline Root Mean Squared Error: {np.sqrt(mse)}")

In [ ]:
# Hyperparameter search: random search with 5-fold CV on the training set only
param_dist = {
    'n_estimators': randint(100, 800),
    'learning_rate': loguniform(0.01, 0.3),
    'max_depth': randint(2, 8),
    'min_child_weight': randint(1, 50),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 5),
    'reg_alpha': loguniform(1e-3, 10),
    'reg_lambda': loguniform(1e-2, 50),
}

search = RandomizedSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=1),
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
search.fit(X_train, y_train)

print(f"Best CV RMSE: {-search.best_score_:.4f}")
print(search.best_params_)

In [ ]:
# Evaluate the tuned model on the held-out test set
best_model = search.best_estimator_
tuned_pred = best_model.predict(X_test)
print(f"Tuned test RMSE:    {np.sqrt(mean_squared_error(y_test, tuned_pred)):.4f}")
print(f"Baseline test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

In [ ]:
data['xgb_pred'] = best_model.predict(data[features])
data.head()

In [ ]:
# Baseline: existing xScore vs actual Score
rmse = np.sqrt(np.mean((data['Score'] - data['xScore']) ** 2))
print(f"xScore RMSE: {rmse:.4f}")